In [1]:
import pandas as pd
import numpy as np

In [2]:
def combine_stats(df):
    df = df.copy()
    df.columns = df.columns.str.upper()
    df['W1COLUMN'] = df['WSCORE']
    df['W2COLUMN'] = df['LSCORE']

    # Winning rows: SEASON, DAYNUM, all W-prefixed cols, WON=1
    w_cols = ['SEASON', 'DAYNUM'] + [c for c in df.columns if c.startswith('W')]
    winning = df[w_cols].copy()
    winning['WON'] = 1
    rename_map = {
        c: c[1:] for c in winning.columns
        if c.startswith('W') and c not in ['WON', 'WLOC', 'W1COLUMN', 'W2COLUMN']
    }
    winning = winning.rename(columns=rename_map)

    # Losing rows: SEASON, DAYNUM, WLOC, W1COLUMN, W2COLUMN, all L-prefixed cols, WON=0
    l_cols = ['SEASON', 'DAYNUM', 'WLOC', 'W1COLUMN', 'W2COLUMN'] + [
        c for c in df.columns if c.startswith('L')
    ]
    losing = df[l_cols].copy()
    losing['WON'] = 0
    rename_map = {
        c: c[1:] for c in losing.columns
        if c.startswith('L') and c not in ['WON', 'WLOC']
    }
    losing = losing.rename(columns=rename_map)

    return pd.concat([winning, losing], ignore_index=True)

In [3]:
m_reg = combine_stats(pd.read_csv('data_2026/MRegularSeasonDetailedResults.csv'))

In [4]:
m_reg

,SEASON,DAYNUM,TEAMID,SCORE,WLOC,FGM,FGA,FGM3,FGA3,FTM,...,OR,DR,AST,TO,STL,BLK,PF,W1COLUMN,W2COLUMN,WON
0,2003,10,1104,68,N,27,58,3,14,11,...,14,24,13,23,7,1,22,68,62,1
1,2003,10,1272,70,N,26,62,8,20,10,...,15,28,16,13,4,4,18,70,63,1
2,2003,11,1266,73,N,24,58,8,18,17,...,17,26,15,10,5,2,25,73,61,1
3,2003,11,1296,56,N,18,38,3,9,17,...,6,19,11,12,14,2,18,56,50,1
4,2003,11,1400,77,N,30,61,6,14,11,...,17,22,12,14,4,4,20,77,71,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249053,2026,132,1463,84,N,31,71,9,24,13,...,17,24,23,9,9,4,16,88,84,0
249054,2026,132,1276,72,N,30,64,7,24,5,...,11,22,15,7,2,3,18,80,72,0
249055,2026,132,1455,55,N,19,56,4,19,13,...,12,27,8,14,7,7,19,70,55,0
249056,2026,132,1173,62,N,23,53,7,21,9,...,8,31,16,11,3,4,18,70,62,0


In [5]:
w_margin = (
    m_reg[m_reg.WON == 1]
    .assign(SCOREDIFF=lambda x: x.W1COLUMN - x.W2COLUMN)
    .groupby(['SEASON', 'TEAMID'])
    .agg(WINMARGINMEDIAN=('SCOREDIFF', 'median'), WINMARGINMEAN=('SCOREDIFF', 'mean'))
    .reset_index()
)

l_margin = (
    m_reg[m_reg.WON == 0]
    .assign(SCOREDIFF=lambda x: x.W1COLUMN - x.W2COLUMN)
    .groupby(['SEASON', 'TEAMID'])
    .agg(LOSEMARGINMEDIAN=('SCOREDIFF', 'median'), LOSEMARGINMEAN=('SCOREDIFF', 'mean'))
    .reset_index()
)

m_season_margin = w_margin.merge(l_margin, on=['SEASON', 'TEAMID'])
m_reg = m_reg.drop(columns=['W1COLUMN', 'W2COLUMN'])

In [6]:
num_cols = [
    c for c in m_reg.drop(columns=['DAYNUM', 'SEASON', 'TEAMID', 'WLOC']).columns
    if m_reg[c].dtype in [np.float64, np.int64, np.float32, np.int32]
]

agg_dict = {}
for col in num_cols:
    agg_dict[f'{col}_MEAN'] = (col, 'mean')
    agg_dict[f'{col}_MEDIAN'] = (col, 'median')
    agg_dict[f'{col}_STDDEV'] = (col, 'std')

season_stats = (
    m_reg.drop(columns=['DAYNUM'])
    .groupby(['SEASON', 'TEAMID'])
    .agg(**agg_dict)
    .reset_index()
)

# Drop aggregated stat columns for WON, SEASON, TEAMID (not meaningful as averages)
cols_to_drop = [
    c for c in season_stats.columns
    if c.startswith('WON_') or c.startswith('SEASON_') or c.startswith('TEAMID_')
]
season_stats = season_stats.drop(columns=cols_to_drop)

In [7]:
hna = (
    m_reg.groupby(['SEASON', 'TEAMID', 'WLOC'])
    .agg(WINCOUNT=('WON', 'sum'))
    .reset_index()
)
hna['WLOC'] = 'WLOC' + hna['WLOC']
hna = (
    hna.pivot_table(index=['SEASON', 'TEAMID'], columns='WLOC', values='WINCOUNT', fill_value=0)
    .reset_index()
)
hna.columns.name = None

In [8]:
season_joined = (
    season_stats
    .merge(hna, on=['SEASON', 'TEAMID'])
    .merge(m_season_margin, on=['SEASON', 'TEAMID'])
    .drop_duplicates()
    .fillna(0)
)

In [9]:
season_joined[(season_joined.SEASON == 2021) & (season_joined.TEAMID == 1288)]

,SEASON,TEAMID,SCORE_MEAN,SCORE_MEDIAN,SCORE_STDDEV,FGM_MEAN,FGM_MEDIAN,FGM_STDDEV,FGA_MEAN,FGA_MEDIAN,...,PF_MEAN,PF_MEDIAN,PF_STDDEV,WLOCA,WLOCH,WLOCN,WINMARGINMEDIAN,WINMARGINMEAN,LOSEMARGINMEDIAN,LOSEMARGINMEAN
6351,2021,1288,78.263158,79.0,11.584674,26.315789,26.0,4.447879,62.421053,63.0,...,21.368421,22.0,3.435283,7.0,3.0,2.0,9.0,11.083333,7.0,6.285714


In [10]:
conf_games = pd.read_csv('data_2026/MConferenceTourneyGames.csv')
conf_games.columns = conf_games.columns.str.upper()

# Get the conference champion (last game = highest DAYNUM per SEASON, CONFABBREV)
conf_wins = (
    conf_games
    .sort_values('DAYNUM', ascending=False)
    .groupby(['SEASON', 'CONFABBREV'], as_index=False)
    .first()
    .drop(columns=['DAYNUM', 'LTEAMID', 'CONFABBREV'])
    .assign(WON_CONFERENCE=1)
    .rename(columns={'WTEAMID': 'TEAMID'})
)

final = season_joined.merge(conf_wins, on=['SEASON', 'TEAMID'], how='left')
final['WON_CONFERENCE'] = final['WON_CONFERENCE'].fillna(0)
final['TOTAL_WINS'] = final['WLOCN'] + final['WLOCH'] + final['WLOCA']

In [11]:
final[(final.SEASON == 2024) & (final.TEAMID == 1314)]

,SEASON,TEAMID,SCORE_MEAN,SCORE_MEDIAN,SCORE_STDDEV,FGM_MEAN,FGM_MEDIAN,FGM_STDDEV,FGA_MEAN,FGA_MEDIAN,...,PF_STDDEV,WLOCA,WLOCH,WLOCN,WINMARGINMEDIAN,WINMARGINMEAN,LOSEMARGINMEDIAN,LOSEMARGINMEAN,WON_CONFERENCE,TOTAL_WINS
7447,2024,1314,81.470588,80.0,10.866226,28.088235,28.0,4.69279,62.5,61.0,...,4.67023,8.0,14.0,5.0,13.0,15.518519,4.0,5.285714,0.0,27.0


In [12]:
# season is the materialized season stats dataframe (same as final at this point)
season = final.copy()

In [13]:
seeds = pd.read_csv('data_2026/MNCAATourneySeeds.csv')
seeds.columns = seeds.columns.str.upper()
seeds

,SEASON,SEED,TEAMID
0,1985,W01,1207
1,1985,W02,1210
2,1985,W03,1228
3,1985,W04,1260
4,1985,W05,1374
...,...,...,...
2689,2026,Z12,1219
2690,2026,Z13,1218
2691,2026,Z14,1244
2692,2026,Z15,1474


In [14]:
seeds['REGION'] = seeds['SEED'].str[0]
seeds['SEED'] = seeds['SEED'].str[1:].str.replace('[a-z]', '', regex=True).astype(int)
seed_value = seeds[['SEASON', 'TEAMID', 'REGION', 'SEED']].copy()
seed_value

,SEASON,TEAMID,REGION,SEED
0,1985,1207,W,1
1,1985,1210,W,2
2,1985,1228,W,3
3,1985,1260,W,4
4,1985,1374,W,5
...,...,...,...,...
2689,2026,1219,Z,12
2690,2026,1218,Z,13
2691,2026,1244,Z,14
2692,2026,1474,Z,15


In [15]:
tourney = pd.read_csv('data_2026/MNCAATourneyCompactResults.csv')
tourney.columns = tourney.columns.str.upper()
tourney = tourney[['SEASON', 'WTEAMID', 'LTEAMID', 'WSCORE', 'LSCORE', 'DAYNUM']]
tourney

,SEASON,WTEAMID,LTEAMID,WSCORE,LSCORE,DAYNUM
0,1985,1116,1234,63,54,136
1,1985,1120,1345,59,58,136
2,1985,1207,1250,68,43,136
3,1985,1229,1425,58,55,136
4,1985,1242,1325,49,38,136
...,...,...,...,...,...,...
2580,2025,1120,1277,70,64,146
2581,2025,1222,1397,69,50,146
2582,2025,1196,1120,79,73,152
2583,2025,1222,1181,70,67,152


In [16]:
conditions = [
    (tourney.DAYNUM >= 134) & (tourney.DAYNUM <= 135),
    (tourney.DAYNUM >= 136) & (tourney.DAYNUM <= 137),
    (tourney.DAYNUM >= 138) & (tourney.DAYNUM <= 139),
    (tourney.DAYNUM >= 143) & (tourney.DAYNUM <= 144),
    tourney.DAYNUM == 145,
    tourney.DAYNUM == 152,
]
choices = [0, 1, 2, 3, 4, 5]
tourney_round = tourney.copy()
tourney_round['ROUND'] = np.select(conditions, choices, default=6)
tourney_round = tourney_round.drop(columns=['DAYNUM'])
tourney_round

,SEASON,WTEAMID,LTEAMID,WSCORE,LSCORE,ROUND
0,1985,1116,1234,63,54,1
1,1985,1120,1345,59,58,1
2,1985,1207,1250,68,43,1
3,1985,1229,1425,58,55,1
4,1985,1242,1325,49,38,1
...,...,...,...,...,...,...
2580,2025,1120,1277,70,64,6
2581,2025,1222,1397,69,50,6
2582,2025,1196,1120,79,73,5
2583,2025,1222,1181,70,67,5


In [17]:
## Add in conference names, uppercase column headers and values and one hot encode
conf = pd.read_csv('data_2026/MTeamConferences.csv')
conf.columns = conf.columns.str.upper()
conf['CONFABBREV'] = conf['CONFABBREV'].str.upper().str.replace(r'[^a-zA-Z0-9]+', '_', regex=True)
conf = conf.rename(columns={'SEASON': 'C_SEASON', 'TEAMID': 'C_TEAMID'})
conf

,C_SEASON,C_TEAMID,CONFABBREV
0,1985,1102,WAC
1,1985,1103,OVC
2,1985,1104,SEC
3,1985,1106,SWAC
4,1985,1108,SWAC
...,...,...,...
13748,2026,1477,SOUTHLAND
13749,2026,1478,NEC
13750,2026,1479,NEC
13751,2026,1480,A_SUN


In [18]:
tourney_conf_w = (
    tourney_round
    .merge(conf, left_on=['WTEAMID', 'SEASON'], right_on=['C_TEAMID', 'C_SEASON'])
    .drop(columns=['C_SEASON', 'C_TEAMID'])
    .rename(columns={'CONFABBREV': 'W_CONF'})
)
tourney_conf_w

,SEASON,WTEAMID,LTEAMID,WSCORE,LSCORE,ROUND,W_CONF
0,1985,1116,1234,63,54,1,SWC
1,1985,1120,1345,59,58,1,SEC
2,1985,1207,1250,68,43,1,BIG_EAST
3,1985,1229,1425,58,55,1,MVC
4,1985,1242,1325,49,38,1,BIG_EIGHT
...,...,...,...,...,...,...,...
2580,2025,1120,1277,70,64,6,SEC
2581,2025,1222,1397,69,50,6,BIG_TWELVE
2582,2025,1196,1120,79,73,5,SEC
2583,2025,1222,1181,70,67,5,BIG_TWELVE


In [19]:
tourney_conf_round = (
    tourney_conf_w
    .merge(conf, left_on=['LTEAMID', 'SEASON'], right_on=['C_TEAMID', 'C_SEASON'])
    .drop(columns=['C_SEASON', 'C_TEAMID'])
    .rename(columns={'CONFABBREV': 'L_CONF'})
)
tourney_conf_round

,SEASON,WTEAMID,LTEAMID,WSCORE,LSCORE,ROUND,W_CONF,L_CONF
0,1985,1116,1234,63,54,1,SWC,BIG_TEN
1,1985,1120,1345,59,58,1,SEC,BIG_TEN
2,1985,1207,1250,68,43,1,BIG_EAST,ECC
3,1985,1229,1425,58,55,1,MVC,PAC_TEN
4,1985,1242,1325,49,38,1,BIG_EIGHT,MAC
...,...,...,...,...,...,...,...,...
2580,2025,1120,1277,70,64,6,SEC,BIG_TEN
2581,2025,1222,1397,69,50,6,BIG_TWELVE,SEC
2582,2025,1196,1120,79,73,5,SEC,SEC
2583,2025,1222,1181,70,67,5,BIG_TWELVE,ACC


In [20]:
tourney_conf_round

,SEASON,WTEAMID,LTEAMID,WSCORE,LSCORE,ROUND,W_CONF,L_CONF
0,1985,1116,1234,63,54,1,SWC,BIG_TEN
1,1985,1120,1345,59,58,1,SEC,BIG_TEN
2,1985,1207,1250,68,43,1,BIG_EAST,ECC
3,1985,1229,1425,58,55,1,MVC,PAC_TEN
4,1985,1242,1325,49,38,1,BIG_EIGHT,MAC
...,...,...,...,...,...,...,...,...
2580,2025,1120,1277,70,64,6,SEC,BIG_TEN
2581,2025,1222,1397,69,50,6,BIG_TWELVE,SEC
2582,2025,1196,1120,79,73,5,SEC,SEC
2583,2025,1222,1181,70,67,5,BIG_TWELVE,ACC


In [21]:
# Add winner seeds
w_t = (
    tourney_conf_round
    .merge(seed_value, left_on=['SEASON', 'WTEAMID'], right_on=['SEASON', 'TEAMID'])
    .drop(columns=['TEAMID'])
    .rename(columns={'REGION': 'W_REGION', 'SEED': 'W_SEED'})
)

# Add loser seeds
tourney_conf_round = (
    w_t
    .merge(seed_value, left_on=['SEASON', 'LTEAMID'], right_on=['SEASON', 'TEAMID'])
    .drop(columns=['TEAMID'])
    .rename(columns={'REGION': 'L_REGION', 'SEED': 'L_SEED'})
)

In [22]:
tourney_conf_round

,SEASON,WTEAMID,LTEAMID,WSCORE,LSCORE,ROUND,W_CONF,L_CONF,W_REGION,W_SEED,L_REGION,L_SEED
0,1985,1116,1234,63,54,1,SWC,BIG_TEN,X,9,X,8
1,1985,1120,1345,59,58,1,SEC,BIG_TEN,Z,11,Z,6
2,1985,1207,1250,68,43,1,BIG_EAST,ECC,W,1,W,16
3,1985,1229,1425,58,55,1,MVC,PAC_TEN,Y,9,Y,8
4,1985,1242,1325,49,38,1,BIG_EIGHT,MAC,Z,3,Z,14
...,...,...,...,...,...,...,...,...,...,...,...,...
2580,2025,1120,1277,70,64,6,SEC,BIG_TEN,Y,1,Y,2
2581,2025,1222,1397,69,50,6,BIG_TWELVE,SEC,X,1,X,2
2582,2025,1196,1120,79,73,5,SEC,SEC,Z,1,Y,1
2583,2025,1222,1181,70,67,5,BIG_TWELVE,ACC,X,1,W,1


In [23]:
season_w = season.copy()
season_w.columns = ['W_' + c for c in season_w.columns]

season_l = season.copy()
season_l.columns = ['L_' + c for c in season_l.columns]

In [24]:
final = (
    tourney_conf_round
    .merge(
        season_w,
        left_on=['WTEAMID', 'SEASON'],
        right_on=['W_TEAMID', 'W_SEASON'],
    )
    .drop(columns=['W_TEAMID', 'W_SEASON'])
    .merge(
        season_l,
        left_on=['LTEAMID', 'SEASON'],
        right_on=['L_TEAMID', 'L_SEASON'],
    )
    .drop(columns=['L_TEAMID', 'L_SEASON'])
)
final

,SEASON,WTEAMID,LTEAMID,WSCORE,LSCORE,ROUND,W_CONF,L_CONF,W_REGION,W_SEED,...,L_PF_STDDEV,L_WLOCA,L_WLOCH,L_WLOCN,L_WINMARGINMEDIAN,L_WINMARGINMEAN,L_LOSEMARGINMEDIAN,L_LOSEMARGINMEAN,L_WON_CONFERENCE,L_TOTAL_WINS
0,2003,1421,1411,92,84,0,BIG_SOUTH,SWAC,X,16,...,4.557298,4.0,11.0,3.0,5.5,9.055556,7.0,8.666667,1.0,18.0
1,2003,1112,1436,80,51,1,PAC_TEN,AEC,Z,1,...,4.047440,7.0,9.0,3.0,10.0,12.052632,7.5,9.400000,1.0,19.0
2,2003,1113,1272,84,71,1,PAC_TEN,CUSA,Z,10,...,4.339644,7.0,14.0,2.0,11.0,12.695652,3.0,6.666667,0.0,23.0
3,2003,1141,1166,79,73,1,MAC,MVC,Z,11,...,3.115212,7.0,17.0,5.0,19.0,17.793103,7.0,6.000000,1.0,29.0
4,2003,1143,1301,76,74,1,PAC_TEN,ACC,W,8,...,4.212734,3.0,13.0,2.0,11.5,15.055556,11.5,11.583333,0.0,18.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1431,2025,1120,1277,70,64,6,SEC,BIG_TEN,Y,1,...,3.041381,7.0,16.0,4.0,13.0,14.481481,5.0,5.166667,0.0,27.0
1432,2025,1222,1397,69,50,6,BIG_TWELVE,SEC,X,1,...,3.735459,6.0,17.0,4.0,15.0,16.962963,5.0,8.571429,0.0,27.0
1433,2025,1196,1120,79,73,5,SEC,SEC,Z,1,...,3.639535,8.0,14.0,6.0,13.5,17.964286,6.0,6.600000,0.0,28.0
1434,2025,1222,1181,70,67,5,BIG_TWELVE,ACC,X,1,...,4.209424,10.0,18.0,3.0,23.0,23.258065,5.0,4.666667,1.0,31.0


In [25]:
final = pd.get_dummies(final, columns=['W_CONF', 'L_CONF'], drop_first=True, dtype=int)

In [26]:
wloc_cols = ['W_WLOCN', 'W_WLOCH', 'W_WLOCA', 'L_WLOCN', 'L_WLOCH', 'L_WLOCA']
final[wloc_cols] = final[wloc_cols].astype(int)

### This table is all season data joined with historic tournament data

In [27]:
final.to_csv('data_2026/final_features.csv', index=False)

### Create season table for predicting 2026

In [28]:
conf_simple = pd.read_csv('data_2026/MTeamConferences.csv')
conf_simple.columns = conf_simple.columns.str.upper()
conf_simple['CONFABBREV'] = conf_simple['CONFABBREV'].str.upper().str.replace(r'[^a-zA-Z0-9]+', '_', regex=True)

season = season.merge(
    conf_simple[['SEASON', 'TEAMID', 'CONFABBREV']],
    on=['SEASON', 'TEAMID'],
).rename(columns={'CONFABBREV': 'CONF'})

season = pd.get_dummies(season, columns=['CONF'], drop_first=True, dtype=int)

In [29]:
# Save season stats for all teams (not filtered by seeds) so future seasons are included
season.to_csv('data_2026/final_season_stats.csv', index=False)